# ML-08 - Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sinagdevv/flyrank-ml-internship-sinagdev/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook compares interpretable probability rankings with the established Week 4 rule baseline on the same authorized starter data, target, metric, and client-grouped holdout.

## 1. Method choice and why

The decision is which pages should enter a limited human refresh-review queue. I compare Logistic Regression (readable coefficients), a shallow Decision Tree (readable rules), and a Random Forest (a stronger but less transparent interaction model) against the existing Week 4 baseline score. Each classifier's probability is used as a ranking score, not as an automatic edit decision. The target remains the current observed proxy `is_declining_label = (trend_direction == 'down')`; it is not a future decline outcome. I will prefer the simplest method unless a more complex model earns a meaningful Precision@50 improvement on the same holdout.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

repo_root = Path.cwd()
if not (repo_root / "data/processed/refresh_feature_vector.csv").exists():
    repo_root = (repo_root / "../..").resolve()
feature_path = repo_root / "data/processed/refresh_feature_vector.csv"
baseline_path = repo_root / "data/processed/baseline_refresh_queue.csv"
assert feature_path.exists() and baseline_path.exists()
frame = pd.read_csv(feature_path)
baseline = pd.read_csv(baseline_path)
feature_columns = ["impressions_90d", "days_since_last_update", "avg_position", "ctr", "engagement_rate"]
forbidden = {"content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"}
assert set(feature_columns).isdisjoint(forbidden)
assert set(["content_id", "client_id", "is_declining_label"]) <= set(frame.columns)
print(f"Loaded {len(frame):,} rows; candidate feature count={len(feature_columns)}.")

Loaded 30,000 rows; candidate feature count=5.


## 2. Split design

I reuse the Week 4 deterministic client-grouped holdout: 20% of pseudonymized clients are held out with random seed 42. Pages from a client cannot appear in both partitions, so client-specific patterns cannot leak across evaluation. The evaluation population and target are unchanged from the established pipeline. All imputation, scaling, and categorical encoding are inside each model pipeline and fit on training rows only. The holdout is used once for comparison, not tuning.

In [2]:
RANDOM_STATE = 42
all_clients = frame["client_id"].drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
test_clients = set(rng.permutation(all_clients)[:max(1, int(round(len(all_clients) * 0.2)))])
test_mask = frame["client_id"].isin(test_clients)
train_idx = np.flatnonzero(~test_mask.to_numpy())
test_idx = np.flatnonzero(test_mask.to_numpy())
train_clients = set(frame.iloc[train_idx]["client_id"])
holdout_clients = set(frame.iloc[test_idx]["client_id"])
assert train_clients.isdisjoint(holdout_clients)
assert frame.iloc[train_idx]["is_declining_label"].nunique() == 2
assert frame.iloc[test_idx]["is_declining_label"].nunique() == 2
print(f"Split: {len(train_idx):,} train rows / {len(test_idx):,} holdout rows; clients={len(train_clients)}/{len(holdout_clients)}; overlap={len(train_clients & holdout_clients)}")

Split: 27,675 train rows / 2,325 holdout rows; clients=26/6; overlap=0


## 3. Train + compare vs my baseline

The primary metric is Precision@50: the observed-decline fraction among the top 50 holdout pages ranked by each method's score. The baseline score is read from the existing Week 4 queue and evaluated on the identical holdout rows. Supporting metrics are average precision and holdout precision at a 0.5 classification threshold. These are predictive-association results for the current proxy, not causal refresh effects.

In [3]:
X = frame[feature_columns]
y = frame["is_declining_label"].astype(int)
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), feature_columns)
])
models = {
    "logistic_regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE),
    "decision_tree": DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
}

def precision_at_50(y_true, scores):
    order = np.argsort(-np.asarray(scores))[:50]
    return float(np.asarray(y_true)[order].mean())

baseline_lookup = baseline.set_index("content_id")["baseline_refresh_score"]
baseline_scores = frame.iloc[test_idx]["content_id"].map(baseline_lookup).fillna(0).to_numpy()
rows = [{"method": "week4_baseline_rules", "strategy": "client_holdout", "precision_at_50": precision_at_50(y.iloc[test_idx], baseline_scores), "average_precision": average_precision_score(y.iloc[test_idx], baseline_scores), "holdout_precision": precision_score(y.iloc[test_idx], baseline_scores >= 0.5, zero_division=0)}]
trained = {}
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
    scores = pipe.predict_proba(X.iloc[test_idx])[:, 1]
    trained[name] = (pipe, scores)
    rows.append({"method": name, "strategy": "client_holdout", "precision_at_50": precision_at_50(y.iloc[test_idx], scores), "average_precision": average_precision_score(y.iloc[test_idx], scores), "holdout_precision": precision_score(y.iloc[test_idx], scores >= 0.5, zero_division=0)})
comparison = pd.DataFrame(rows)
comparison["difference_from_baseline"] = comparison["precision_at_50"] - comparison.loc[0, "precision_at_50"]
print(f"Evaluation population: {len(test_idx):,} holdout rows; metric top-k=50.")
display(comparison.round(3))

Evaluation population: 2,325 holdout rows; metric top-k=50.


,method,strategy,precision_at_50,average_precision,holdout_precision,difference_from_baseline
0,week4_baseline_rules,client_holdout,0.24,0.468,0.499,0.00
1,logistic_regression,client_holdout,0.22,0.317,0.209,-0.02
2,decision_tree,client_holdout,0.68,0.532,0.516,0.44
3,random_forest,client_holdout,0.76,0.600,0.565,0.52


## 4. Errors and interpretation

I select the model by holdout Precision@50, then inspect its false positives and false negatives. Feature influence is descriptive rather than causal: coefficients or tree importance show what the fitted model used, not what causes decline. A more complex model is justified only if its validated queue precision improves enough to matter to the editor's limited review capacity. Limitations include the current-state proxy, one client holdout, possible instability from the small high-priority queue, and the absence of a future outcome window. Human review remains required.

In [4]:
best_name = comparison.sort_values(["precision_at_50", "average_precision"], ascending=False).iloc[0]["method"]
model_name = best_name if best_name != "week4_baseline_rules" else "random_forest"
pipe, scores = trained.get(model_name, trained["random_forest"])
pred = (scores >= 0.5).astype(int)
errors = frame.iloc[test_idx][["content_id", "client_id", "impressions_90d", "days_since_last_update", "trend_direction"]].copy()
errors["score"] = scores
errors["actual_decline"] = y.iloc[test_idx].to_numpy()
errors["predicted_decline"] = pred
false_positives = errors[(errors.actual_decline == 0) & (errors.predicted_decline == 1)].head(3)
false_negatives = errors[(errors.actual_decline == 1) & (errors.predicted_decline == 0)].head(3)
print(f"Selected method by holdout Precision@50: {model_name}")
print("False-positive examples (pseudonymous IDs only):")
display(false_positives)
print("False-negative examples (pseudonymous IDs only):")
display(false_negatives)
if model_name in trained:
    estimator = pipe.named_steps["model"]
    influence = pd.Series(np.abs(estimator.coef_[0]) if hasattr(estimator, "coef_") else estimator.feature_importances_, index=feature_columns).sort_values(ascending=False).head(3)
    print("Top descriptive feature influences:")
    display(influence.rename("importance").to_frame())
assert train_clients.isdisjoint(holdout_clients)
assert not set(feature_columns).intersection(forbidden)

Selected method by holdout Precision@50: random_forest
False-positive examples (pseudonymous IDs only):
False-negative examples (pseudonymous IDs only):
Top descriptive feature influences:


,content_id,client_id,impressions_90d,days_since_last_update,trend_direction,score,actual_decline,predicted_decline
147,content_bec800684271,client_f74efabef1,56,20,up,0.509547,0,1
168,content_d7cbd76b788d,client_f74efabef1,17992,20,stable,0.614905,0,1
192,content_a0b5322afbb0,client_d4735e3a26,34,20,up,0.558726,0,1


,content_id,client_id,impressions_90d,days_since_last_update,trend_direction,score,actual_decline,predicted_decline
48,content_326fa2fa449f,client_98a3ab7c34,4,1,down,0.241652,1,0
63,content_d5d3c2e98937,client_f74efabef1,63,20,down,0.494140,1,0
141,content_0af426466565,client_f74efabef1,9,8,down,0.440017,1,0


,importance
impressions_90d,0.381871
avg_position,0.293819
days_since_last_update,0.188841


## Self-check

- [x] Method choice, split, metric, baseline, and evaluation population are explicit.
- [x] Preprocessing is inside the model pipeline and fit on training rows only.
- [x] IDs, `trend_direction`, `trend_pct`, and future information are excluded from features.
- [x] Client overlap is asserted to be zero and random seed is fixed.
- [x] Model-versus-baseline results and error examples use executed holdout outputs.
- [x] Limitations distinguish current decline ranking from future prediction and causality.
- [ ] Commit and submit to the internship portal (intentionally not performed in this task).